# Different gender classifications of Australian and Canadian author names

## Load libraries and data

In [1]:
# import libraries
import pickle
import statistics
import matplotlib.pyplot as plt
import re
import pandas as pd
import numpy as np
from collections import Counter
from collections import OrderedDict
from collections import defaultdict
import math
import json
import requests

In [2]:
# load in author-name files
aus_authors_in = '../data/aus/author_analysis/aus_authors_29jan2026.pkl'
with open(aus_authors_in, 'rb') as f:
    names_aus = pickle.load(f, encoding="utf-8")
    
can_authors_in = '../data/canada/author_analysis/can_authors_29jan2026.pkl'
with open(can_authors_in, 'rb') as f:
    names_can = pickle.load(f, encoding="utf-8")

# 1. Namsor

In [ ]:
# Namsor API key
api_key = "INSERT YOUR NAMSOR API KEY HERE"
headers = {
  "X-API-KEY": api_key,
 "Accept": "application/json"
}

In [ ]:
# function to make API call
def namsor_call(fullname, country_code): # Country code = CA or AU
    fullname_formatted = fullname.strip().replace(' ', '%')
    url = f"https://v2.namsor.com/NamSorAPIv2/api2/json/genderFullGeo/{fullname}/{country_code}"
    response = requests.request("GET", url, headers=headers)
    classify_dict = json.loads(response.text)
    print(k)
    return classify_dict

### 1.1 Canada

In [ ]:
# Create dictionaries
namsor_can = {}
namsor_final_can = {}

In [ ]:
# Namsor classification of Canada names
for k in names_can.keys():
    if k not in namsor_can.keys():
        classify = namsor_call(names_can[k]['full'], 'CA')
        namsor_can[k] = classify

In [ ]:
# Save raw Namsor classification outputs to file
out_can = '../data/canada/author_analysis/can_namsor_raw_outputs.pkl'
with open(out_can, 'wb') as f:
    pickle.dump(namsor_can, f)

In [ ]:
# Open raw Namsor Canada classifications file
in_can_namsor = '../data/canada/author_analysis/can_namsor_raw_outputs.pkl'
with open(in_can_namsor, 'rb') as f:
    namsor_can_in = pickle.load(f, encoding="utf-8")

In [ ]:
# Convert raw Namsor outputs to cleaned gender classifications
for k in namsor_can_in.keys():
    if namsor_can_in[k]['probabilityCalibrated'] >= 0.6:
        namsor_final_can[k] = namsor_can_in[k]['likelyGender']
    else:
        print(namsor_can_in[k]['name'])
        namsor_final_can[k] = 'unknown/other'

In [ ]:
# Save cleaned classifications to file
out_can2 = './author_analysis/namsor_can_final_29jan2026.pkl'
with open(out_can2, 'wb') as f:
    pickle.dump(namsor_final_can, f)

### 1.2 Australia

In [ ]:
# Create dictionaries
namsor_aus = {}
namsor_final_aus = {}

In [ ]:
# Namsor classification of Australia names
for k in names_aus.keys():
    if k not in namsor_aus.keys():
        classify = namsor_call(names_aus[k]['full'], 'AU')
        namsor_aus[k] = classify

In [ ]:
# Save raw Namsor classification outputs to file
out_aus = '../data/aus/author_analysis/aus_namsor_raw_outputs.pkl'
with open(out_aus, 'wb') as f:
    pickle.dump(namsor_aus, f)

In [ ]:
# Open raw Namsor Canada classifications file
in_aus_namsor = '../data/aus/author_analysis/aus_namsor_raw_outputs.pkl'

with open(in_aus_namsor, 'rb') as f:
    namsor_aus_in = pickle.load(f, encoding="utf-8")

In [ ]:
# Convert raw Namsor outputs to cleaned gender classifications
for k in namsor_aus_in.keys():
    if namsor_aus_in[k]['probabilityCalibrated'] >= 0.6:
        namsor_final_aus[k] = namsor_aus_in[k]['likelyGender']
    else:
        print(namsor_aus_in[k]['name'])
        namsor_final_aus[k] = 'unknown/other'

In [ ]:
# Save cleaned classifications to file
out_aus2 = './author_analysis/namsor_aus_final_29jan2026.pkl'
with open(out_aus2, 'wb') as f:
    pickle.dump(namsor_final_aus, f)

# 2. gender_guesser classifications

In [29]:
def gender_guesser_full(authors):
    
    gen_guesser_authors = {k: None for k in authors.keys()}
    
    import gender_guesser.detector as gender
    d = gender.Detector()
    
    for k in authors.keys():
        name = authors[k]['first']
        x = d.get_gender(name.capitalize())
        if x == 'female' or x == 'mostly_female':
            gen_guesser_authors[k] = 'female'
        elif x == 'male' or x == 'mostly_male':
            gen_guesser_authors[k] = 'male'
        else: # androgyneous or unknown
            gen_guesser_authors[k] = 'unknown/other'
    
    return gen_guesser_authors    

### 2.2 Canada

In [30]:
gender_guesser_only_can = gender_guesser_full(names_can)

In [34]:
# Save cleaned classifications to file
with open('../data/canada/author_analysis/can_gender_guesser.pkl', 'wb') as f:
    pickle.dump(gender_guesser_only_can, f)

### 2.1 Australia

In [35]:
gender_guesser_only_aus = gender_guesser_full(names_aus)

In [38]:
# Save cleaned classifications to file
with open('../data/aus/author_analysis/aus_gender_guesser.pkl', 'wb') as f:
    pickle.dump(gender_guesser_only_aus, f)

# 3. Metadata-based classifications

### 3.1 Canada

In [3]:
# read in canadian baby names
with open('../data/canada/author_analysis/can_baby_names.pkl', 'rb') as f:
    can_baby_names = pickle.load(f, encoding="utf-8")

In [43]:
# for each canadian name, assign it the canadian baby name gender if its in that dictionary
# otherwise assign in a None vaslue

can_metadata_gen = {}
for k in names_can.keys():
    if names_can[k]['first'] in can_baby_names.keys():
        can_metadata_gen[k] = can_baby_names[names_can[k]['first']]
    else:
        can_metadata_gen[k] = None

In [44]:
print(len([k for k in can_metadata_gen.keys() if can_metadata_gen[k] == 'female']))
print(len([k for k in can_metadata_gen.keys() if can_metadata_gen[k] == 'male']))
print(len([k for k in can_metadata_gen.keys() if can_metadata_gen[k] == None]))
print(len(can_metadata_gen))

860
976
198
2034


In [45]:
for k in can_metadata_gen.keys():
    if names_can[k]['first'] not in can_baby_names.keys() and can_metadata_gen[k] != None:
        print(k)

In [47]:
# Save classifications to file
with open('../data/canada/author_analysis/can_metadata_only_gen.pkl', 'wb') as f:
    pickle.dump(can_metadata_gen, f)

### 3.2 Australia 

female: mrs, ms, miss, misss, sister
male: mr, mister, mstr, sir
unknown/other: mx
non-gendered: dr, capt, captain,..., me, mt, sr, 

In [4]:
def aus_gender(authors):
    male_t = ['mr', 'mstr', 'mister', 'sir']
    female_t = ['ms', 'miss', 'mrs', 'misss', 'sister']
    other_t = ['mx']
    author_gen = {}
    none_counter = 0
    for k in authors.keys():
        if authors[k]['title'] in male_t: 
            gen_val = 'male'
        elif authors[k]['title'] in female_t:
            gen_val = 'female'
        elif authors[k]['title'] in other_t:
            gen_val = 'unknown/other'
        else:
            gen_val = None
            none_counter += 1
        author_gen[k] = gen_val
        
    print(f'Number of None values - Australian gender categorisation from titles: {none_counter}')
    return author_gen

In [5]:
metadata_aus_gen = aus_gender(names_aus)

Number of None values - Australian gender categorisation from titles: 535


For repeat authors who have some gender classifications of 'male'/'female' and some of None, if all gender classification are the same single value (.e.g. all 'female'), we want to assign this gender classification to the None instances of the name. These cases arise when an author writes several petitions but does not always use a gendered title with their name (e.g. sometimes uses 'Ms' and sometimes uses no title). 

In [6]:
names_aus2 = {k:names_aus[k]['full'] for k in names_aus.keys()}

# Invert the dictionary to group keys into a list
grouped_dict = defaultdict(list)
for key, value in names_aus2.items():
    grouped_dict[value].append(key)

result = dict(grouped_dict) # for each full name, list of petition IDs associated to it
#print(result)

# for each full name, list of gender classifications associated to it
result2 = {k: [metadata_aus_gen[v] for v in result[k]] for k in result.keys()}
#print(result2)

# want to update names with only 'gender' and None values:
multi_gen_update = {}
for k in result2.keys():
    if len(result2[k]) > 1 and len(set(result2[k])) > 1 and None in result2[k]:
        single_gendered_title = [x for x in set(result2[k]) if x != None]
        if len(single_gendered_title) == 1:
            multi_gen_update[k] = single_gendered_title[0]
            
#print(multi_gen_update)

result_to_update = {k:v for k,v in result.items() if k in multi_gen_update.keys()} # getting the list of ids for each name in multi_gen_updates

#print(result_to_update)

In [1]:
# update the metadata-gender labels to account for repeat authors sometimes using a gendered title 
# and sometimes not using a title or using a gender-unknown title

metadata_aus_gen_updated = metadata_aus_gen.copy()

for k in result_to_update.keys(): # for each full author name in the dictionary
    #print(k)
    #print(multi_gen_update[k])
    for x in result_to_update[k]: # for each petition ID associated to the full author name 'k'
        metadata_aus_gen_updated[x] = multi_gen_update[k]

NameError: name 'metadata_aus_gen' is not defined

In [9]:
print(Counter(metadata_aus_gen.values()))
print(Counter(metadata_aus_gen_updated.values()))

Counter({'male': 2026, 'female': 1337, None: 535, 'unknown/other': 8})
Counter({'male': 2115, 'female': 1365, None: 418, 'unknown/other': 8})


In [10]:
# Save classifications to file
with open('../data/aus/author_analysis/aus_metadata_only_gen.pkl', 'wb') as f:
    pickle.dump(metadata_aus_gen_updated, f)

In [11]:
# result2 = {k: [metadata_aus_gen_updated[v] for v in result[k]] for k in result.keys()}
# for k in result2.keys():
#     if len(result2[k]) > 2 and len(set(result2[k])) > 1:
#         print(k)
#         print(set(result2[k]))

In [12]:
pets_with_gen = [names_aus[k]['full'] for k in metadata_aus_gen_updated.keys() if metadata_aus_gen_updated[k] != None]
print(len(pets_with_gen))
print(len(set(pets_with_gen)))

3488
2613


In [13]:
#set(pets_with_gen)

### 3.3. Create ID lists of Australian names that are also in Canada baby names lists for validation of method

To validate the Namsor classifications, we want a list of names with 'ground-truth' labels. These are all names with a gendered title in the Australian dataset. We sort these names into two groups based on whether or not the first name of the author appears in the Canada baby names collections. To deduplicate the data, we will kep track of not only full names but genders and only add IDs associated to unique name-gender combinations.

In [14]:
# removing duplicates - keeping one instance of each full-name-to-gender-classification combination
# for almost all full names with a ground-truth gender label, there is only one gender label associated to it

metadata_aus_in_can = {'in': [], 'out': []}
names_in_dict = {} # want to deduplicate instances of FULL NAMES (bc different ppl with same first name could have different ground-truth gender labels)
for k in names_aus.keys():
    if metadata_aus_gen_updated[k] != None: # names with a groud-truth gender label
        if names_aus[k]['full'] not in names_in_dict.keys(): # full name has not been seen
            if names_aus[k]['first'] in can_baby_names.keys():
                metadata_aus_in_can['in'].append(k)
            else:
                metadata_aus_in_can['out'].append(k)
            names_in_dict[names_aus[k]['full']] = [metadata_aus_gen_updated[k]]
        elif metadata_aus_gen_updated[k] not in names_in_dict[names_aus[k]['full']]: # full name has been seen, but this gender categorisation of the name has not been seen
            if names_aus[k]['first'] in can_baby_names.keys():
                metadata_aus_in_can['in'].append(k)
            else:
                metadata_aus_in_can['out'].append(k)
            names_in_dict[names_aus[k]['full']].append(metadata_aus_gen_updated[k])
    

        

In [16]:
# for k in names_in_dict.keys():
#     if len(names_in_dict[k]) > 1:
#         print(k)

In [17]:
sum([len(names_in_dict[k]) for k in names_in_dict.keys()])

2614

In [18]:
len(metadata_aus_in_can['in'])  + len(metadata_aus_in_can['out'])

2614

In [21]:
# write dictionary of in-canada and out-canada Australian validation petition IDs to file

with open('../data/aus/author_analysis/aus_namsor_validation_ids.pkl', 'wb') as f:
    pickle.dump(metadata_aus_in_can, f)